In [8]:
import torch

print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA Available: True
GPU: Tesla T4


In [227]:
!pip install -q --upgrade \
langgraph \
langchain \
langchain-community \
langchain-openai \
langchain-anthropic \
qdrant-client \
sentence-transformers \
transformers \
spacy \
boto3 \
openai \
anthropic \
openai-whisper \
pymupdf \
pypdf \
structlog \
python-dotenv \
mlflow \
dvc \
peft \
trl \
bitsandbytes \
accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 246.2/246.2 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.9/132.9 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.4/596.4 kB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 74.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 17.1 MB/s eta 0:00:00


In [10]:
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 82.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [1]:
import torch
import fitz
import whisper
import spacy
import qdrant_client
import langgraph
import transformers

print("All imports successful")

All imports successful


In [2]:
import os

PROJECT_ROOT = "/content/LexGraphAI"

folders = [
    "data/raw",
    "data/processed",
    "data/chunks",
    "data/annotations",
    "models",
    "outputs",
    "src/ocr",
    "src/nlp",
    "src/rag",
    "src/agents",
    "src/transcription",
    "src/evaluation",
    "src/prompts",
    "src/utils",
    "experiments",
    "mlruns"
]

for folder in folders:
    os.makedirs(
        os.path.join(PROJECT_ROOT, folder),
        exist_ok=True
    )

print("Project structure created")

Project structure created


In [3]:
import os

print(os.listdir(PROJECT_ROOT))

['experiments', 'models', 'src', 'mlruns', 'outputs', 'data']


In [4]:
os.listdir(f"{PROJECT_ROOT}/data")

['chunks', 'annotations', 'processed', 'raw']

In [5]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [6]:
!mkdir -p "/content/drive/MyDrive/LexGraphAI"

In [7]:
from google.colab import files

uploaded = files.upload()

Saving the_constitution_of_india.pdf to the_constitution_of_india.pdf


In [8]:
filename = next(iter(uploaded))

print(filename)

the_constitution_of_india.pdf


In [9]:
import shutil
import os

source_path = filename

destination_path = os.path.join(
    PROJECT_ROOT,
    "data/raw",
    filename
)

shutil.copy(source_path, destination_path)

print(destination_path)

/content/LexGraphAI/data/raw/the_constitution_of_india.pdf


In [10]:
os.listdir(f"{PROJECT_ROOT}/data/raw")

['the_constitution_of_india.pdf']

In [11]:
import fitz

pdf = fitz.open(destination_path)

print("Pages:", len(pdf))

Pages: 256


In [12]:
first_page = pdf.load_page(0)

sample_text = first_page.get_text()

print("Characters detected:", len(sample_text.strip()))

Characters detected: 147


In [13]:
if len(sample_text.strip()) > 100:
    pdf_type = "digital"
else:
    pdf_type = "scanned"

print("PDF Type:", pdf_type)

PDF Type: digital


In [14]:
pdf_type == "digital"

True

In [15]:
def extract_digital_pdf(path: str) -> list[dict]:
    document = fitz.open(path)

    pages = []

    for page_number in range(len(document)):

        page = document.load_page(page_number)

        text = page.get_text()

        pages.append(
            {
                "page": page_number + 1,
                "text": text
            }
        )

    return pages

In [16]:
pages = extract_digital_pdf(destination_path)

In [17]:
print(pages[0]["text"][:1000])

 
 
 
 
 
THE CONSTITUTION OF INDIA 
[As on 9th December, 2020] 
 
 
 
 
 
2020 
 
 
GOVERNMENT OF INDIA 
MINISTRY OF LAW AND JUSTICE 
LEGISLATIVE DEPARTMENT



In [18]:
if pdf_type == "scanned":
    raise NotImplementedError(
        "OCR pipeline will be implemented in Step 6."
    )

In [19]:
document = {
    "filename": filename,
    "pdf_type": pdf_type,
    "total_pages": len(pages),
    "pages": pages
}

In [20]:
document.keys()

dict_keys(['filename', 'pdf_type', 'total_pages', 'pages'])

In [21]:
import json

output_path = os.path.join(
    PROJECT_ROOT,
    "data/processed",
    "document.json"
)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(
        document,
        f,
        ensure_ascii=False,
        indent=2
    )

print(output_path)

/content/LexGraphAI/data/processed/document.json


In [22]:
with open(output_path, "r", encoding="utf-8") as f:
    loaded = json.load(f)

print(loaded["filename"])
print(loaded["total_pages"])

the_constitution_of_india.pdf
256


In [23]:
print(loaded["pages"][0]["text"][:500])

 
 
 
 
 
THE CONSTITUTION OF INDIA 
[As on 9th December, 2020] 
 
 
 
 
 
2020 
 
 
GOVERNMENT OF INDIA 
MINISTRY OF LAW AND JUSTICE 
LEGISLATIVE DEPARTMENT



In [24]:
{
  "filename": "petition.pdf",
  "pdf_type": "digital",
  "total_pages": 12,
  "pages": [
    {
      "page": 1,
      "text": "IN THE HIGH COURT ..."
    }
  ]
}

{'filename': 'petition.pdf',
 'pdf_type': 'digital',
 'total_pages': 12,
 'pages': [{'page': 1, 'text': 'IN THE HIGH COURT ...'}]}

In [25]:
import json
import os

document_path = os.path.join(
    PROJECT_ROOT,
    "data/processed",
    "document.json"
)

with open(document_path, "r", encoding="utf-8") as f:
    document = json.load(f)

print(document["filename"])
print(document["total_pages"])

the_constitution_of_india.pdf
256


In [26]:
page_texts = []

for page in document["pages"]:
    page_texts.append(
        {
            "page": page["page"],
            "text": page["text"]
        }
    )

print(len(page_texts))

256


In [27]:
import re


def clean_text(text: str) -> str:

    text = re.sub(r"\n{2,}", "\n", text)

    text = re.sub(r"\s+", " ", text)

    text = text.strip()

    return text

In [28]:
sample = clean_text(page_texts[0]["text"])

print(sample[:500])

THE CONSTITUTION OF INDIA [As on 9th December, 2020] 2020 GOVERNMENT OF INDIA MINISTRY OF LAW AND JUSTICE LEGISLATIVE DEPARTMENT


In [29]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [30]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=[
        "\n\n",
        "\n",
        ". ",
        " "
    ]
)

In [31]:
chunks = []

chunk_id = 0

for page in page_texts:

    cleaned = clean_text(page["text"])

    split_chunks = splitter.split_text(cleaned)

    for text_chunk in split_chunks:

        chunks.append(
            {
                "chunk_id": chunk_id,
                "page": page["page"],
                "text": text_chunk
            }
        )

        chunk_id += 1

In [32]:
print("Total chunks:", len(chunks))

Total chunks: 1177


In [33]:
chunks[0]

{'chunk_id': 0,
 'page': 1,
 'text': 'THE CONSTITUTION OF INDIA [As on 9th December, 2020] 2020 GOVERNMENT OF INDIA MINISTRY OF LAW AND JUSTICE LEGISLATIVE DEPARTMENT'}

In [34]:
print(chunks[0]["text"])

THE CONSTITUTION OF INDIA [As on 9th December, 2020] 2020 GOVERNMENT OF INDIA MINISTRY OF LAW AND JUSTICE LEGISLATIVE DEPARTMENT


In [35]:
for chunk in chunks:

    chunk["source_document"] = document["filename"]

    chunk["pdf_type"] = document["pdf_type"]

In [36]:
chunks[0]

{'chunk_id': 0,
 'page': 1,
 'text': 'THE CONSTITUTION OF INDIA [As on 9th December, 2020] 2020 GOVERNMENT OF INDIA MINISTRY OF LAW AND JUSTICE LEGISLATIVE DEPARTMENT',
 'source_document': 'the_constitution_of_india.pdf',
 'pdf_type': 'digital'}

In [37]:
chunks_path = os.path.join(
    PROJECT_ROOT,
    "data/chunks",
    "chunks.json"
)

with open(chunks_path, "w", encoding="utf-8") as f:
    json.dump(
        chunks,
        f,
        ensure_ascii=False,
        indent=2
    )

print(chunks_path)

/content/LexGraphAI/data/chunks/chunks.json


In [38]:
with open(chunks_path, "r", encoding="utf-8") as f:
    loaded_chunks = json.load(f)

print(len(loaded_chunks))

1177


In [39]:
loaded_chunks[0]

{'chunk_id': 0,
 'page': 1,
 'text': 'THE CONSTITUTION OF INDIA [As on 9th December, 2020] 2020 GOVERNMENT OF INDIA MINISTRY OF LAW AND JUSTICE LEGISLATIVE DEPARTMENT',
 'source_document': 'the_constitution_of_india.pdf',
 'pdf_type': 'digital'}

In [40]:
import json
import os

chunks_path = os.path.join(
    PROJECT_ROOT,
    "data/chunks",
    "chunks.json"
)

with open(chunks_path, "r", encoding="utf-8") as f:
    chunks = json.load(f)

print("Chunks loaded:", len(chunks))

Chunks loaded: 1177


In [41]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "BAAI/bge-small-en-v1.5"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [42]:
texts = [chunk["text"] for chunk in chunks]

In [43]:
embeddings = embedding_model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)

Batches:   0%|          | 0/37 [00:00<?, ?it/s]

In [44]:
embeddings.shape

(1177, 384)

In [45]:
from qdrant_client import QdrantClient

client = QdrantClient(":memory:")

In [46]:
from qdrant_client.models import (
    VectorParams,
    Distance
)

VECTOR_SIZE = embeddings.shape[1]

client.create_collection(
    collection_name="legal_documents",
    vectors_config=VectorParams(
        size=VECTOR_SIZE,
        distance=Distance.COSINE
    )
)

True

In [47]:
from qdrant_client.models import PointStruct

points = []

for idx, chunk in enumerate(chunks):

    points.append(
        PointStruct(
            id=idx,
            vector=embeddings[idx].tolist(),
            payload=chunk
        )
    )

In [48]:
len(points)

1177

In [49]:
client.upsert(
    collection_name="legal_documents",
    points=points
)

UpdateResult(operation_id=0, status=<UpdateStatus.COMPLETED: 'completed'>)

In [50]:
collection = client.get_collection(
    "legal_documents"
)

print(collection.points_count)

1177


In [51]:
def retrieve(
    query: str,
    top_k: int = 5
):

    query_embedding = embedding_model.encode(
        query,
        normalize_embeddings=True
    )

    results = client.query_points(
        collection_name="legal_documents",
        query=query_embedding.tolist(),
        limit=top_k
    )

    return results

In [52]:
results = retrieve(
    "What relief is sought by the petitioner?"
)

In [53]:
retrieved_points = results.points

for result in retrieved_points:

    print("-" * 80)

    print("Score:", round(result.score, 3))

    print("Page:", result.payload["page"])

    print(result.payload["text"][:500])

--------------------------------------------------------------------------------
Score: 0.654
Page: 35
35 and endow them with such powers and authority as may be necessary to enable them to function as units of self-government. 41. Right to work, to education and to public assistance in certain cases.—The State shall, within the limits of its economic capacity and development, make effective provision for securing the right to work, to education and to public assistance in cases of unemployment, old age, sickness and disablement, and in other cases of undeserved want. 42. Provision for just and h
--------------------------------------------------------------------------------
Score: 0.654
Page: 129
129 (b) where the authority empowered to dismiss or remove a person or to reduce him in rank is satisfied that for some reason, to be recorded by that authority in writing, it is not reasonably practicable to hold such inquiry; or (c) where the President or the Governor, as the case may be, 

In [54]:
from transformers import pipeline

generator = pipeline(
    task="text-generation",
    model="microsoft/Phi-3-mini-4k-instruct",
    device_map="auto"
)

config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/16.5k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.44k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.94M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

In [60]:
from typing import TypedDict


class GraphState(TypedDict):
    question: str
    retrieved_chunks: list
    context: str
    answer: str
    citations: list
    confidence: float

In [61]:
def retrieval_agent(state: GraphState):

    results = retrieve(
        query=state["question"],
        top_k=5
    )

    chunks = []

    citations = []

    for result in results.points:

        chunks.append(result.payload["text"])

        citations.append(
            {
                "page": result.payload["page"],
                "chunk_id": result.payload["chunk_id"]
            }
        )

    state["retrieved_chunks"] = chunks

    state["citations"] = citations

    state["context"] = "\n\n".join(chunks)

    return state

In [62]:
retrieval_agent(
    {
        "question": "What relief is sought by the petitioner?",
        "retrieved_chunks": [],
        "context": "",
        "answer": "",
        "citations": [],
        "confidence": 0.0
    }
)

{'question': 'What relief is sought by the petitioner?',
 'retrieved_chunks': ['35 and endow them with such powers and authority as may be necessary to enable them to function as units of self-government. 41. Right to work, to education and to public assistance in certain cases.—The State shall, within the limits of its economic capacity and development, make effective provision for securing the right to work, to education and to public assistance in cases of unemployment, old age, sickness and disablement, and in other cases of undeserved want. 42. Provision for just and humane conditions of work and maternity relief.—The State shall make provision for securing just and humane conditions of work and for maternity relief. 43',
  '129 (b) where the authority empowered to dismiss or remove a person or to reduce him in rank is satisfied that for some reason, to be recorded by that authority in writing, it is not reasonably practicable to hold such inquiry; or (c) where the President or th

In [63]:
SYSTEM_PROMPT = """
You are an Indian legal research assistant.

Rules:

1. Answer only using the supplied context.
2. Never invent facts.
3. Never fabricate citations.
4. If evidence is insufficient, say so.
5. Use precise legal language.
"""

In [64]:
def reasoning_agent(state: GraphState):

    prompt = f"""
{SYSTEM_PROMPT}

Question:
{state['question']}

Context:
{state['context']}

Answer:
"""

    response = generator(
        prompt,
        max_new_tokens=256,
        do_sample=False
    )

    generated_text = response[0]["generated_text"]

    answer = generated_text.split("Answer:")[-1].strip()

    state["answer"] = answer

    return state

In [65]:
def verification_agent(state: GraphState):

    if len(state["citations"]) == 0:

        state["confidence"] = 0.0

        return state

    confidence = min(
        1.0,
        len(state["citations"]) / 5
    )

    state["confidence"] = confidence

    return state

In [66]:
from langgraph.graph import (
    StateGraph,
    END
)

In [67]:
workflow = StateGraph(GraphState)

In [68]:
workflow.add_node(
    "retrieve",
    retrieval_agent
)

workflow.add_node(
    "reason",
    reasoning_agent
)

workflow.add_node(
    "verify",
    verification_agent
)

In [69]:
workflow.set_entry_point("retrieve")

workflow.add_edge(
    "retrieve",
    "reason"
)

workflow.add_edge(
    "reason",
    "verify"
)

workflow.add_edge(
    "verify",
    END
)

In [70]:
graph = workflow.compile()

In [71]:
result = graph.invoke(
    {
        "question": (
            "What relief is sought by the petitioner?"
        ),
        "retrieved_chunks": [],
        "context": "",
        "answer": "",
        "citations": [],
        "confidence": 0.0
    }
)

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


In [72]:
print(result["answer"])

The relief sought by the petitioner is not explicitly mentioned in the provided context. The context discusses various legal provisions related to the administration of autonomous councils, the right to work, education, public assistance, just and humane conditions of work, maternity relief, and the regulation of essential goods and rent control. However, without additional information about the specific petition or the issues raised by the petitioner, it is not possible to determine the exact relief sought. If the context provided is insufficient to ascertain the relief sought by the petitioner, it should be stated as such.



You are an Indian legal research assistant.

Rules:

1. Answer only using the supplied context.
2. Never invent facts.
3. Never fabricate citations.
4. If evidence is insufficient, say so.
5. Use precise legal language.
6. Do not use any legal terms that are not defined in the context.
7. Avoid using any legal terms that are not directly related to the petitione

In [73]:
for citation in result["citations"]:

    print(
        f"Page {citation['page']} | "
        f"Chunk {citation['chunk_id']}"
    )

Page 35 | Chunk 114
Page 129 | Chunk 595
Page 196 | Chunk 941
Page 206 | Chunk 998
Page 134 | Chunk 629


In [75]:
print(
    f"Confidence: "
    f"{result['confidence']:.2f}"
)

Confidence: 1.00


In [149]:
def ask_lexgraph(
    question: str,
    graph
):

    state = {
        "question": question,
        "retrieved_chunks": [],
        "context": "",
        "answer": "",
        "citations": [],
        "confidence": 0.0
    }

    return graph.invoke(state)

In [150]:
response = ask_lexgraph(
    "What are the allegations against the respondent?",
    graph
)

print(response["answer"])

AttributeError: 'DiGraph' object has no attribute 'invoke'

In [151]:
import spacy

nlp = spacy.load("en_core_web_sm")

print("spaCy loaded")

spaCy loaded


In [152]:
from transformers import (
    AutoTokenizer,
    AutoModel
)

MODEL_NAME = "nlpaueb/legal-bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

legal_bert = AutoModel.from_pretrained(
    MODEL_NAME
)

print("Legal-BERT loaded")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: nlpaueb/legal-bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Legal-BERT loaded


In [153]:
import json
import os

chunks_path = os.path.join(
    PROJECT_ROOT,
    "data/chunks",
    "chunks.json"
)

with open(chunks_path, "r", encoding="utf-8") as f:
    chunks = json.load(f)

print(len(chunks))

1177


In [154]:
import re

SECTION_PATTERN = re.compile(
    r"Section\s+\d+[A-Za-z\-]*",
    re.IGNORECASE
)

ACT_PATTERN = re.compile(
    r"(Code of Civil Procedure|CPC|Limitation Act|FEMA|IBC)",
    re.IGNORECASE
)

CASE_PATTERN = re.compile(
    r"\b[A-Z]{1,5}\s*No\.?\s*\d+\/\d{4}\b",
    re.IGNORECASE
)

COURT_PATTERN = re.compile(
    r"(Supreme Court of India|High Court.*?|District Court.*?)",
    re.IGNORECASE
)

In [155]:
def extract_legal_entities(text: str):

    entities = []

    doc = nlp(text)

    for ent in doc.ents:

        entities.append(
            {
                "text": ent.text,
                "label": ent.label_
            }
        )

    for match in SECTION_PATTERN.finditer(text):

        entities.append(
            {
                "text": match.group(),
                "label": "SECTION"
            }
        )

    for match in ACT_PATTERN.finditer(text):

        entities.append(
            {
                "text": match.group(),
                "label": "ACT"
            }
        )

    for match in CASE_PATTERN.finditer(text):

        entities.append(
            {
                "text": match.group(),
                "label": "CASE_NUMBER"
            }
        )

    for match in COURT_PATTERN.finditer(text):

        entities.append(
            {
                "text": match.group(),
                "label": "COURT"
            }
        )

    return entities

In [156]:
sample_chunk = chunks[0]["text"]

entities = extract_legal_entities(
    sample_chunk
)

entities

[{'text': '9th December', 'label': 'DATE'},
 {'text': '2020', 'label': 'DATE'},
 {'text': '2020', 'label': 'DATE'}]

In [157]:
all_entities = []

for chunk in chunks:

    extracted = extract_legal_entities(
        chunk["text"]
    )

    for entity in extracted:

        entity["chunk_id"] = chunk["chunk_id"]

        entity["page"] = chunk["page"]

        entity["source_document"] = (
            chunk["source_document"]
        )

        all_entities.append(entity)

In [158]:
print(len(all_entities))

17803


In [159]:
unique = {}

for entity in all_entities:

    key = (
        entity["text"],
        entity["label"],
        entity["page"]
    )

    unique[key] = entity

all_entities = list(unique.values())

In [160]:
entities_path = os.path.join(
    PROJECT_ROOT,
    "data/processed",
    "entities.json"
)

with open(
    entities_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        all_entities,
        f,
        ensure_ascii=False,
        indent=2
    )

print(entities_path)

/content/LexGraphAI/data/processed/entities.json


In [161]:
all_entities[:10]

[{'text': '9th December',
  'label': 'DATE',
  'chunk_id': 0,
  'page': 1,
  'source_document': 'the_constitution_of_india.pdf'},
 {'text': '2020',
  'label': 'DATE',
  'chunk_id': 0,
  'page': 1,
  'source_document': 'the_constitution_of_india.pdf'},
 {'text': 'Article',
  'label': 'LAW',
  'chunk_id': 1,
  'page': 2,
  'source_document': 'the_constitution_of_india.pdf'},
 {'text': '″ Clause',
  'label': 'ORG',
  'chunk_id': 1,
  'page': 2,
  'source_document': 'the_constitution_of_india.pdf'},
 {'text': 'C.O. ″ Constitution Order',
  'label': 'PRODUCT',
  'chunk_id': 1,
  'page': 2,
  'source_document': 'the_constitution_of_india.pdf'},
 {'text': '″ Inserted',
  'label': 'ORG',
  'chunk_id': 1,
  'page': 2,
  'source_document': 'the_constitution_of_india.pdf'},
 {'text': '″ Page',
  'label': 'ORG',
  'chunk_id': 1,
  'page': 2,
  'source_document': 'the_constitution_of_india.pdf'},
 {'text': '″ Repealed',
  'label': 'PERSON',
  'chunk_id': 1,
  'page': 2,
  'source_document': 'the_co

In [162]:
def search_entities(
    label: str = None
):

    if label is None:
        return all_entities

    return [
        entity
        for entity in all_entities
        if entity["label"] == label
    ]

In [163]:
import json
import os

entities_path = os.path.join(
    PROJECT_ROOT,
    "data/processed",
    "entities.json"
)

with open(entities_path, "r", encoding="utf-8") as f:
    entities = json.load(f)

print("Entities loaded:", len(entities))

Entities loaded: 10495


In [164]:
chunks_path = os.path.join(
    PROJECT_ROOT,
    "data/chunks",
    "chunks.json"
)

with open(chunks_path, "r", encoding="utf-8") as f:
    chunks = json.load(f)

print("Chunks loaded:", len(chunks))

Chunks loaded: 1177


In [165]:
chunk_lookup = {
    chunk["chunk_id"]: chunk
    for chunk in chunks
}

In [166]:
chunk_lookup[0].keys()

dict_keys(['chunk_id', 'page', 'text', 'source_document', 'pdf_type'])

In [167]:
import re

SECTION_ACT_PATTERN = re.compile(
    r"(Section\s+\d+[A-Za-z\-]*)\s+of\s+the\s+([^.,;\n]+)",
    re.IGNORECASE
)

CASE_CITATION_PATTERN = re.compile(
    r"([A-Za-z\s]+)\s+v\.?\s+([A-Za-z\s]+)",
    re.IGNORECASE
)

In [168]:
relationships = []

for chunk in chunks:

    text = chunk["text"]

    for match in SECTION_ACT_PATTERN.finditer(text):

        section = match.group(1).strip()

        act = match.group(2).strip()

        relationships.append(
            {
                "source": section,
                "target": act,
                "relation": "BELONGS_TO",
                "page": chunk["page"],
                "chunk_id": chunk["chunk_id"]
            }
        )

    for match in CASE_CITATION_PATTERN.finditer(text):

        petitioner = match.group(1).strip()

        respondent = match.group(2).strip()

        relationships.append(
            {
                "source": petitioner,
                "target": respondent,
                "relation": "VERSUS",
                "page": chunk["page"],
                "chunk_id": chunk["chunk_id"]
            }
        )

In [169]:
print("Relationships:", len(relationships))

Relationships: 67


In [170]:
entity_index = {}

for entity in entities:

    key = entity["text"].lower()

    if key not in entity_index:
        entity_index[key] = []

    entity_index[key].append(entity)

In [171]:
entity_index.get("section 34", [])

[]

In [172]:
!pip install -q networkx

In [173]:
import networkx as nx

In [174]:
knowledge_graph = nx.DiGraph()

In [175]:
for entity in entities:

    knowledge_graph.add_node(
        entity["text"],
        label=entity["label"]
    )

In [176]:
for relation in relationships:

    knowledge_graph.add_edge(
        relation["source"],
        relation["target"],
        relation=relation["relation"],
        page=relation["page"]
    )

In [177]:
print("Nodes:", knowledge_graph.number_of_nodes())
print("Edges:", knowledge_graph.number_of_edges())

Nodes: 3469
Edges: 45


In [178]:
relations_path = os.path.join(
    PROJECT_ROOT,
    "data/processed",
    "relationships.json"
)

with open(relations_path, "w") as f:

    json.dump(
        relationships,
        f,
        indent=2
    )

print(relations_path)

/content/LexGraphAI/data/processed/relationships.json


In [179]:
graph_path = os.path.join(
    PROJECT_ROOT,
    "data/processed",
    "knowledge_graph.graphml"
)

nx.write_graphml(
    knowledge_graph,
    graph_path
)

print(graph_path)

/content/LexGraphAI/data/processed/knowledge_graph.graphml


In [180]:
def hybrid_retrieve(
    query: str,
    top_k: int = 5
):

    vector_results = retrieve(
        query=query,
        top_k=top_k
    )

    matched_entities = []

    query_lower = query.lower()

    for key in entity_index:

        if key in query_lower:

            matched_entities.extend(
                entity_index[key]
            )

    return {
        "vector_results": vector_results,
        "matched_entities": matched_entities
    }

In [181]:
result = hybrid_retrieve(
    "What does Section 34 of CPC say?"
)

print(result["matched_entities"])

[{'text': '3', 'label': 'CARDINAL', 'chunk_id': 2, 'page': 3, 'source_document': 'the_constitution_of_india.pdf'}, {'text': '3', 'label': 'CARDINAL', 'chunk_id': 4, 'page': 4, 'source_document': 'the_constitution_of_india.pdf'}, {'text': '3', 'label': 'CARDINAL', 'chunk_id': 55, 'page': 23, 'source_document': 'the_constitution_of_india.pdf'}, {'text': '3', 'label': 'CARDINAL', 'chunk_id': 67, 'page': 25, 'source_document': 'the_constitution_of_india.pdf'}, {'text': '3', 'label': 'CARDINAL', 'chunk_id': 71, 'page': 26, 'source_document': 'the_constitution_of_india.pdf'}, {'text': '3', 'label': 'CARDINAL', 'chunk_id': 78, 'page': 27, 'source_document': 'the_constitution_of_india.pdf'}, {'text': '3', 'label': 'CARDINAL', 'chunk_id': 83, 'page': 28, 'source_document': 'the_constitution_of_india.pdf'}, {'text': '3', 'label': 'CARDINAL', 'chunk_id': 87, 'page': 29, 'source_document': 'the_constitution_of_india.pdf'}, {'text': '3', 'label': 'CARDINAL', 'chunk_id': 91, 'page': 30, 'source_docu

In [182]:
evaluation_set = [
    {
        "question": "What relief is sought by the petitioner?",
        "expected_pages": [4, 5],
        "expected_keywords": [
            "relief",
            "petitioner"
        ]
    },
    {
        "question": "Which court is hearing the matter?",
        "expected_pages": [1],
        "expected_keywords": [
            "court"
        ]
    },
    {
        "question": "What sections are referenced?",
        "expected_pages": [3, 4],
        "expected_keywords": [
            "section"
        ]
    }
]

In [183]:
def recall_at_k(
    retrieved_pages,
    expected_pages
):

    retrieved = set(retrieved_pages)

    expected = set(expected_pages)

    hits = len(
        retrieved.intersection(expected)
    )

    return hits / len(expected)

In [184]:
recall_at_k(
    [1, 4, 5],
    [4, 5]
)

1.0

In [185]:
def reciprocal_rank(
    retrieved_pages,
    expected_pages
):

    for rank, page in enumerate(
        retrieved_pages,
        start=1
    ):

        if page in expected_pages:
            return 1 / rank

    return 0

In [186]:
reciprocal_rank(
    [2, 4, 5],
    [4]
)

0.5

In [187]:
retrieval_results = []

for sample in evaluation_set:

    results = retrieve(
        sample["question"],
        top_k=5
    )

    pages = [
        r.payload["page"]
        for r in results.points
    ]

    recall = recall_at_k(
        pages,
        sample["expected_pages"]
    )

    rr = reciprocal_rank(
        pages,
        sample["expected_pages"]
    )

    retrieval_results.append(
        {
            "question": sample["question"],
            "pages": pages,
            "recall@5": recall,
            "rr": rr
        }
    )

In [188]:
retrieval_results

[{'question': 'What relief is sought by the petitioner?',
  'pages': [35, 129, 196, 206, 134],
  'recall@5': 0.0,
  'rr': 0},
 {'question': 'Which court is hearing the matter?',
  'pages': [62, 65, 63, 88, 64],
  'recall@5': 0.0,
  'rr': 0},
 {'question': 'What sections are referenced?',
  'pages': [2, 177, 243, 208, 26],
  'recall@5': 0.0,
  'rr': 0}]

In [189]:
avg_recall = sum(
    r["recall@5"]
    for r in retrieval_results
) / len(retrieval_results)

mrr = sum(
    r["rr"]
    for r in retrieval_results
) / len(retrieval_results)

print("Recall@5:", round(avg_recall, 3))
print("MRR:", round(mrr, 3))

Recall@5: 0.0
MRR: 0.0


In [190]:
def citation_accuracy(
    predicted_pages,
    expected_pages
):

    predicted = set(predicted_pages)

    expected = set(expected_pages)

    correct = len(
        predicted.intersection(expected)
    )

    return correct / max(
        len(predicted),
        1
    )

In [191]:
citation_accuracy(
    [4, 5],
    [4, 5]
)

1.0

In [145]:
def hallucinated(response):

    if len(response["citations"]) == 0:
        return True

    if response["confidence"] < 0.5:
        return True

    return False

In [146]:
hallucinated(
    {
        "citations": [],
        "confidence": 0.2
    }
)

True

In [193]:
import time

benchmark_results = []

for sample in evaluation_set:

    start = time.time()

    latency = time.time() - start

    predicted_pages = [
        citation["page"]
        for citation in response["citations"]
    ]

    benchmark_results.append(
        {
            "question": sample["question"],
            "latency": latency,
            "citation_accuracy": citation_accuracy(
                predicted_pages,
                sample["expected_pages"]
            ),
            "hallucinated": hallucinated(
                response
            )
        }
    )

In [194]:
benchmark_results

[{'question': 'What relief is sought by the petitioner?',
  'latency': 1.6689300537109375e-06,
  'citation_accuracy': 0.0,
  'hallucinated': False},
 {'question': 'Which court is hearing the matter?',
  'latency': 9.5367431640625e-07,
  'citation_accuracy': 0.0,
  'hallucinated': False},
 {'question': 'What sections are referenced?',
  'latency': 2.384185791015625e-07,
  'citation_accuracy': 0.0,
  'hallucinated': False}]

In [195]:
avg_latency = sum(
    r["latency"]
    for r in benchmark_results
) / len(benchmark_results)

avg_citation_accuracy = sum(
    r["citation_accuracy"]
    for r in benchmark_results
) / len(benchmark_results)

hallucination_rate = sum(
    r["hallucinated"]
    for r in benchmark_results
) / len(benchmark_results)

In [196]:
print(
    f"Average Latency: {avg_latency:.2f}s"
)

print(
    f"Citation Accuracy: "
    f"{avg_citation_accuracy:.2f}"
)

print(
    f"Hallucination Rate: "
    f"{hallucination_rate:.2f}"
)

Average Latency: 0.00s
Citation Accuracy: 0.00
Hallucination Rate: 0.00


In [197]:
import json
import os

report = {
    "retrieval": {
        "recall@5": avg_recall,
        "mrr": mrr
    },
    "generation": {
        "citation_accuracy":
            avg_citation_accuracy,
        "hallucination_rate":
            hallucination_rate
    },
    "performance": {
        "latency_seconds":
            avg_latency
    }
}

In [198]:
report_path = os.path.join(
    PROJECT_ROOT,
    "outputs",
    "benchmark_report.json"
)

with open(report_path, "w") as f:
    json.dump(
        report,
        f,
        indent=2
    )

print(report_path)

/content/LexGraphAI/outputs/benchmark_report.json


In [200]:
BASE_MODEL = "microsoft/Phi-3-mini-4k-instruct"

In [201]:
training_data = [
    {
        "instruction": (
            "Identify the legal provisions."
        ),
        "input": (
            "The petitioner seeks relief "
            "under Section 34 of the "
            "Code of Civil Procedure."
        ),
        "output": (
            "Section 34, Code of Civil Procedure"
        )
    },
    {
        "instruction": (
            "Summarize the legal issue."
        ),
        "input": (
            "The appeal concerns insolvency "
            "proceedings under IBC."
        ),
        "output": (
            "The dispute relates to "
            "insolvency proceedings under IBC."
        )
    }
]

In [202]:
import json
import os

dataset_path = os.path.join(
    PROJECT_ROOT,
    "data/annotations",
    "finetune.json"
)

with open(dataset_path, "w") as f:
    json.dump(training_data, f, indent=2)

print(dataset_path)

/content/LexGraphAI/data/annotations/finetune.json


In [203]:
from datasets import Dataset

dataset = Dataset.from_list(
    training_data
)

dataset

Dataset({
    features: ['instruction', 'input', 'output'],
    num_rows: 2
})

In [204]:
def format_example(example):

    prompt = f"""
### Instruction:
{example["instruction"]}

### Input:
{example["input"]}

### Response:
{example["output"]}
"""

    return {"text": prompt}

In [205]:
dataset = dataset.map(
    format_example
)

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

In [208]:
print(dataset[0]["text"])


### Instruction:
Identify the legal provisions.

### Input:
The petitioner seeks relief under Section 34 of the Code of Civil Procedure.

### Response:
Section 34, Code of Civil Procedure



In [207]:
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig
)

import torch

In [209]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

In [210]:
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL
)

tokenizer.pad_token = tokenizer.eos_token

In [211]:
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto"
)

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

In [218]:
from peft import LoraConfig

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["qkv_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)

In [222]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./legal_qlora",

    per_device_train_batch_size=2,

    gradient_accumulation_steps=4,

    num_train_epochs=3,

    learning_rate=2e-4,

    logging_steps=10,

    save_strategy="epoch",

    fp16=False, # Changed to False to fix the BFloat16 error

    report_to="none"
)

In [223]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,

    args=training_args,

    peft_config=lora_config
)

/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:964: FutureWarning: The default `loss_type` will change from `'nll'` to `'chunked_nll'` in TRL 1.7. For standard models this is transparent (same math, lower memory) and no action is needed — you'll get the new default automatically on upgrade. If you use a custom model, check ahead of time that `loss_type='chunked_nll'` runs and yields the same loss as `'nll'`; if it doesn't, pin `loss_type='nll'` to keep the current behavior and please open an issue at https://github.com/huggingface/trl/issues so we can address the edge case.
  args = SFTConfig(**dict_args)
/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config`

Adding EOS to train dataset:   0%|          | 0/2 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/2 [00:00<?, ? examples/s]

In [224]:
trainer.train()

Step,Training Loss


TrainOutput(global_step=3, training_loss=1.886727015177409, metrics={'train_runtime': 4.662, 'train_samples_per_second': 1.287, 'train_steps_per_second': 0.643, 'total_flos': 8208172118016.0, 'train_loss': 1.886727015177409, 'entropy': 1.5677083333333333, 'mean_token_accuracy': 0.6271186470985413, 'num_tokens': 360.0, 'epoch': 3.0})

In [225]:
save_path = os.path.join(
    PROJECT_ROOT,
    "models",
    "legal_phi3_qlora"
)

trainer.save_model(save_path)

tokenizer.save_pretrained(save_path)

print(save_path)

/content/LexGraphAI/models/legal_phi3_qlora


In [230]:
finetuning_metrics = {
    "base_model": BASE_MODEL,
    "method": "QLoRA",
    "epochs": 3,
    "learning_rate": 2e-4
}

In [231]:
metrics_path = os.path.join(
    PROJECT_ROOT,
    "outputs",
    "finetuning_metrics.json"
)

with open(metrics_path, "w") as f:
    json.dump(
        finetuning_metrics,
        f,
        indent=2
    )

In [232]:
!pip install -q mlflow dvc

In [233]:
import mlflow

print(mlflow.__version__)

3.14.0


In [234]:
import os

os.makedirs(
    f"{PROJECT_ROOT}/mlruns",
    exist_ok=True
)

os.makedirs(
    f"{PROJECT_ROOT}/experiments",
    exist_ok=True
)

In [236]:
import mlflow
import os

os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"

MLFLOW_URI = f"file:{PROJECT_ROOT}/mlruns"

mlflow.set_tracking_uri(
    MLFLOW_URI
)

mlflow.set_experiment(
    "LexGraphAI"
)

2026/06/17 09:44:24 INFO mlflow.tracking.fluent: Experiment with name 'LexGraphAI' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:///content/LexGraphAI/mlruns/867757850496017815', creation_time=1781689464773, effective_trace_archival_retention=None, experiment_id='867757850496017815', last_update_time=1781689464773, lifecycle_stage='active', name='LexGraphAI', tags={}, trace_location=None, workspace='default'>

In [237]:
print(mlflow.get_tracking_uri())

file:/content/LexGraphAI/mlruns


In [238]:
with mlflow.start_run():

    mlflow.log_param(
        "embedding_model",
        "BAAI/bge-small-en-v1.5"
    )

    mlflow.log_param(
        "llm_model",
        BASE_MODEL
    )

    mlflow.log_param(
        "chunk_size",
        1000
    )

    mlflow.log_param(
        "chunk_overlap",
        200
    )

    mlflow.log_metric(
        "recall_at_5",
        avg_recall
    )

    mlflow.log_metric(
        "mrr",
        mrr
    )

    mlflow.log_metric(
        "citation_accuracy",
        avg_citation_accuracy
    )

    mlflow.log_metric(
        "hallucination_rate",
        hallucination_rate
    )

    mlflow.log_metric(
        "latency_seconds",
        avg_latency
    )

print("Experiment logged")

Experiment logged


In [239]:
import json

report = {
    "embedding_model":
        "BAAI/bge-small-en-v1.5",

    "llm_model":
        BASE_MODEL,

    "recall_at_5":
        avg_recall,

    "mrr":
        mrr,

    "citation_accuracy":
        avg_citation_accuracy,

    "hallucination_rate":
        hallucination_rate,

    "latency_seconds":
        avg_latency
}

In [240]:
report_path = os.path.join(
    PROJECT_ROOT,
    "outputs",
    "evaluation_report.json"
)

with open(report_path, "w") as f:
    json.dump(
        report,
        f,
        indent=2
    )

In [241]:
with mlflow.start_run():

    mlflow.log_artifact(
        report_path
    )

    mlflow.log_artifact(
        entities_path
    )

    mlflow.log_artifact(
        relations_path
    )

    mlflow.log_artifact(
        metrics_path
    )

In [242]:
with mlflow.start_run():

    mlflow.log_param(
        "model_type",
        "QLoRA"
    )

    mlflow.log_artifacts(
        save_path,
        artifact_path="model"
    )

In [243]:
runs = mlflow.search_runs()

runs[
    [
        "run_id",
        "metrics.recall_at_5",
        "metrics.citation_accuracy",
        "metrics.hallucination_rate"
    ]
]

,run_id,metrics.recall_at_5,metrics.citation_accuracy,metrics.hallucination_rate
0,bfa08d7b6771493eacc027ce71192478,NaN,NaN,NaN
1,20c10e8e62634724ae8f8a0cd19ba4f8,NaN,NaN,NaN
2,dadaa5f709954863b7c63731e18cb443,0.0,0.0,0.0


In [244]:
dvc_yaml = """
stages:

  ingestion:
    cmd: python src/ingestion.py
    deps:
      - data/raw
    outs:
      - data/processed

  chunking:
    cmd: python src/chunking.py
    deps:
      - data/processed
    outs:
      - data/chunks

  training:
    cmd: python src/train.py
    deps:
      - data/annotations
    outs:
      - models
"""

In [245]:
with open("dvc.yaml", "w") as f:
    f.write(dvc_yaml)

In [246]:
import shutil

backup_dir = (
    "/content/drive/MyDrive/LexGraphAI_Backup"
)

shutil.copytree(
    PROJECT_ROOT,
    backup_dir,
    dirs_exist_ok=True
)

'/content/drive/MyDrive/LexGraphAI_Backup'

In [247]:
summary = {
    "project": "LexGraph AI",

    "embedding_model":
        "BAAI/bge-small-en-v1.5",

    "llm_model":
        BASE_MODEL,

    "retrieval": {
        "recall_at_5":
            avg_recall,

        "mrr":
            mrr
    },

    "generation": {
        "citation_accuracy":
            avg_citation_accuracy,

        "hallucination_rate":
            hallucination_rate
    }
}

In [248]:
summary_path = os.path.join(
    PROJECT_ROOT,
    "outputs",
    "experiment_summary.json"
)

with open(summary_path, "w") as f:
    json.dump(
        summary,
        f,
        indent=2
    )